# Stage 3: Sentiment Analysis (NLP Models)

Fine-tunes every model in `NLP_MODELS` on the weak-labeled KO headlines from Stage 1, repeated across every seed in `CONFIG["random_seeds"]`. The full sweep is `len(NLP_MODELS) × len(random_seeds)` runs (currently 4 × 10 = 40), which is enough to compute mean ± std and run statistical significance tests in Stage 6.

## What this stage produces
- `./results/nlp_results.csv`: one row per (model, seed) with accuracy, F1, train/inference times, and ISO 8601 UTC start/end timestamps for both phases.
- `./artifacts/nlp_probs.parquet`: per-headline 3-class probabilities from the single best (model, seed) combination, used by Stage 4 to build rolling sentiment features for the regression `+NLP` variant.
- `./artifacts/nlp_probs_run_meta.json`: sidecar metadata describing which run produced the probability file and its wall-clock timing windows.

## What this stage consumes
- `./artifacts/text_df.parquet` (weak-labeled headlines, from Stage 1).
- `./artifacts/num_df.parquet` (only used to compute the shared cutoff date).

## Energy join hook
Training and inference are bracketed with both `time.time()` (for fast per-run summaries) and `datetime.now(timezone.utc).isoformat()` (for joining against an external power log post-run). Energy is **not** measured by Python here: a separate wall-meter / RAPL log is integrated over each run's `[wall_*_start_iso, wall_*_end_iso]` window after the pipeline finishes.

## Best-model selection
The "best" run is picked by **F1, not accuracy**, because the weak labels are class-imbalanced (mostly neutral) and accuracy can be inflated by always predicting the majority class. F1 (weighted) penalizes that failure mode.


In [1]:
# Shared helpers, config, and model registries.
from common import *
from datetime import datetime, timezone

# HuggingFace training utilities used in this stage.
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2ForSequenceClassification,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
)

# Weak-labeled KO headlines from Stage 1.
text_df = load_text_df()
# num_df is loaded only to compute a shared split cutoff date.
num_df = load_num_df()

# Date-level precision for clean chronological splits.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# Train/test boundary used by both text and numeric tasks.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)
print(f"Chronological cutoff: {cutoff_date.date()}")

# Per-run metrics collector for CSV export.
results = ResultsCollector()

[common] device=cuda  artifacts=/home/hrilab/energy-analysis-pipeline/artifacts  results=/home/hrilab/energy-analysis-pipeline/results


Chronological cutoff: 2021-06-10


## 3.1 Training routine

In [2]:
def build_tokenizer_and_model(model_path, config):
    """Create tokenizer/model pair for a checkpoint path."""
    # GPT-2 special-case: GPT-2 is currently commented out of NLP_MODELS, but
    # this branch is preserved so that re-enabling GPT-2 is a one-line registry
    # change (uncomment the entries in common.py). GPT-2 needs a pad token
    # (its tokenizer ships without one: reuse EOS) and the corresponding
    # pad_token_id must be set on model.config so attention masks work
    # correctly. This branch is dead-by-design, not dead-by-accident.
    is_gpt2 = "gpt2" in model_path
    if is_gpt2:
        tokenizer = GPT2Tokenizer.from_pretrained(model_path)
        tokenizer.pad_token = tokenizer.eos_token

        model = GPT2ForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        # Non-GPT models can use Auto classes directly.
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )

    # Move model to GPU/CPU selected in common.py.
    model.to(DEVICE)
    return tokenizer, model


def train_nlp_model(model_name, model_path, text_df, config, seed, cutoff_date):
    """Train one model for one seed and return metrics + predictions."""
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=config)

    # Chronological train/test datasets to avoid temporal leakage.
    train_ds, test_ds, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=config["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    # Temporary trainer outputs for this run.
    run_dir = RESULTS_DIR / f"{model_name.replace(' ', '_')}_seed{seed}"
    # eval_strategy="epoch" runs test-set evaluation at the end of each epoch,
    # which gives a learning curve and catches divergence early. Side-effect:
    # `train_time_s` includes those per-epoch evaluations, not just gradient
    # updates. The energy-join numbers (wall_train_start_iso/end_iso) bracket
    # the same span, so external energy is consistent with reported time.
    training_args = TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["train_batch_size"],
        per_device_eval_batch_size=config["eval_batch_size"],
        learning_rate=config["learning_rate"],
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        seed=seed,
        report_to="none",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=config.get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_clf_metrics,
    )

    # ISO timestamps let an external power log (sampled by a wall meter or
    # RAPL outside this notebook) integrate energy over each [start, end]
    # window after the run completes. They are NOT used for internal timing:
    # `time.time()` deltas handle that. Both methods bracket the same code so
    # the windows agree to within a millisecond.
    wall_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    wall_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Evaluate on chronological test split.
    eval_out = trainer.evaluate()

    # Prediction/inference time + wall-clock anchors for external energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    pred_out = trainer.predict(test_ds)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # Convert logits to class IDs via argmax.
    preds = np.argmax(pred_out.predictions, axis=-1)

    # Metrics appended to nlp_results.csv.
    record = {
        "model": model_name,
        "seed": seed,
        "accuracy": eval_out["eval_accuracy"],
        "f1": eval_out["eval_f1"],
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(test_ds),
        "wall_train_start_iso": wall_train_start_iso,
        "wall_train_end_iso": wall_train_end_iso,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }

    # Free memory before the next run.
    del model, trainer
    torch.cuda.empty_cache()

    return record, preds


def train_best_model_and_export_probs(model_name, model_path, seed, text_df, cutoff_date):
    """Retrain best run and export per-headline class probabilities."""
    # The best (model, seed) is retrained from scratch instead of reused from
    # memory because the main sweep above issues `del model, trainer` at the
    # end of every run to free GPU memory between configurations. By the time
    # the winning run is identified, the winning weights are gone.
    #
    # Cost: ~10% extra compute relative to the main sweep (one extra training
    # of the best configuration). A future optimization would checkpoint each
    # trained model to disk during the sweep and reload the winner here.
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=CONFIG["nlp"])

    # Fit using the chronological training slice only.
    train_ds, _, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    output_dir = RESULTS_DIR / f"best_{model_name.replace(' ', '_')}_seed{seed}"
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=CONFIG["nlp"]["epochs"],
        per_device_train_batch_size=CONFIG["nlp"]["train_batch_size"],
        per_device_eval_batch_size=CONFIG["nlp"]["eval_batch_size"],
        learning_rate=CONFIG["nlp"]["learning_rate"],
        save_strategy="no",
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=CONFIG["nlp"].get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
    )
    wall_best_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    trainer.train()
    best_train_time = time.time() - t0
    wall_best_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Score EVERY headline (train slice + test slice), not just test. Stage 4's
    # `+NLP` variant builds rolling-K-day sentiment features by averaging
    # per-headline probabilities over a lookback window, so it needs a
    # probability for every headline date in the analysis window, including
    # train-period headlines that contribute to the rolling average for the
    # earliest test-period rows. Labels are placeholders (zeros) because only
    # the model's softmax output is consumed here, never compared to truth.
    infer_ds = SentimentDataset(
        texts=text_df["text"],
        labels=pd.Series(np.zeros(len(text_df), dtype=int)),
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
    )

    wall_best_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    prediction_output = trainer.predict(infer_ds)
    best_infer_time = time.time() - t0
    wall_best_infer_end_iso = datetime.now(timezone.utc).isoformat()
    logits = prediction_output.predictions
    # Softmax converts logits into class probabilities summing to 1.
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    # One row per headline so Stage 4 can build rolling sentiment features.
    nlp_probs_df = pd.DataFrame(
        {
            "date": text_df["date"].values,
            "prob_negative": probs[:, LABEL_TO_ID["negative"]],
            "prob_neutral": probs[:, LABEL_TO_ID["neutral"]],
            "prob_positive": probs[:, LABEL_TO_ID["positive"]],
            "model": model_name,
            "seed": seed,
        }
    )

    # Sidecar metadata file. The energy-join script reads this to know which
    # (model, seed) produced nlp_probs.parquet and over what wall-clock windows
    # the best-model retrain + scoring happened, so its energy can be
    # attributed separately from the main 40-run sweep recorded in
    # nlp_results.csv.
    import json
    best_run_meta = {
        "model": model_name,
        "seed": seed,
        "wall_best_train_start_iso": wall_best_train_start_iso,
        "wall_best_train_end_iso": wall_best_train_end_iso,
        "best_train_time_s": best_train_time,
        "wall_best_infer_start_iso": wall_best_infer_start_iso,
        "wall_best_infer_end_iso": wall_best_infer_end_iso,
        "best_infer_time_s": best_infer_time,
        "n_headlines_scored": int(len(text_df)),
    }
    with open(ARTIFACTS_DIR / "nlp_probs_run_meta.json", "w") as f:
        json.dump(best_run_meta, f, indent=2)

    # Free memory after export.
    del model, trainer
    torch.cuda.empty_cache()

    return nlp_probs_df

## 3.2 Run every (model, seed) combination

In [3]:
# nlp_predictions: built but never persisted and never read downstream. Kept
# for optional in-notebook diagnostics (e.g., confusion matrices, error
# analysis on specific seeds). Safe to remove if unused.
nlp_predictions = {}

# Train every model across every configured seed.
for model_name, model_path in NLP_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        print(f"\n{'=' * 60}")
        print(f"  {model_name}  |  seed {seed}")
        print(f"{'=' * 60}")

        # Train and evaluate one (model, seed) run.
        record, preds = train_nlp_model(
            model_name=model_name,
            model_path=model_path,
            text_df=text_df,
            config=CONFIG["nlp"],
            seed=seed,
            cutoff_date=cutoff_date,
        )

        # Save metrics and predictions for this run.
        results.add_nlp(record)
        nlp_predictions[(model_name, seed)] = preds

        # Compact run summary.
        print(f"  Accuracy: {record['accuracy']:.4f}  |  F1: {record['f1']:.4f}")
        print(
            f"  Train: {record['train_time_s']:.1f}s  |  "
            f"Infer: {record['infer_time_s']:.1f}s"
        )

# Preview collected NLP metrics.
results.nlp_df().round(6)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



  DistilBERT  |  seed 0


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Accuracy: 0.3747  |  F1: 0.3437
  Train: 47.0s  |  Infer: 1.0s


,model,seed,accuracy,f1,train_time_s,infer_time_s,n_test_samples,wall_train_start_iso,wall_train_end_iso,wall_infer_start_iso,wall_infer_end_iso
0,DistilBERT,0,0.401021,0.342115,26.118155,0.476544,2546,2026-05-07T22:23:54.084088+00:00,2026-05-07T22:24:20.202253+00:00,2026-05-07T22:24:20.622910+00:00,2026-05-07T22:24:21.099460+00:00
1,DistilBERT,1,0.384525,0.350954,25.527247,0.477422,2546,2026-05-07T22:24:21.580606+00:00,2026-05-07T22:24:47.107876+00:00,2026-05-07T22:24:47.540034+00:00,2026-05-07T22:24:48.017463+00:00
2,DistilBERT,2,0.386489,0.345907,25.570949,0.488660,2546,2026-05-07T22:24:48.542874+00:00,2026-05-07T22:25:14.113840+00:00,2026-05-07T22:25:14.548523+00:00,2026-05-07T22:25:15.037191+00:00
3,DistilBERT,3,0.395915,0.346254,25.576559,0.489171,2546,2026-05-07T22:25:15.548364+00:00,2026-05-07T22:25:41.124940+00:00,2026-05-07T22:25:41.575795+00:00,2026-05-07T22:25:42.064972+00:00
4,DistilBERT,4,0.411233,0.344710,25.577692,0.486341,2546,2026-05-07T22:25:42.569695+00:00,2026-05-07T22:26:08.147409+00:00,2026-05-07T22:26:08.599582+00:00,2026-05-07T22:26:09.085931+00:00
5,DistilBERT,5,0.395915,0.340970,25.575190,0.490432,2546,2026-05-07T22:26:09.599970+00:00,2026-05-07T22:26:35.175201+00:00,2026-05-07T22:26:35.641163+00:00,2026-05-07T22:26:36.131601+00:00
6,DistilBERT,6,0.430086,0.279891,25.547640,0.495561,2546,2026-05-07T22:26:36.620919+00:00,2026-05-07T22:27:02.168592+00:00,2026-05-07T22:27:02.626298+00:00,2026-05-07T22:27:03.121865+00:00
7,DistilBERT,7,0.412804,0.334853,25.505496,0.504230,2546,2026-05-07T22:27:03.613554+00:00,2026-05-07T22:27:29.119071+00:00,2026-05-07T22:27:29.581751+00:00,2026-05-07T22:27:30.085988+00:00
8,DistilBERT,8,0.374705,0.333463,25.621577,0.502079,2546,2026-05-07T22:27:30.590046+00:00,2026-05-07T22:27:56.211642+00:00,2026-05-07T22:27:56.687054+00:00,2026-05-07T22:27:57.189139+00:00
9,DistilBERT,9,0.390024,0.334312,25.645869,0.508528,2546,2026-05-07T22:27:57.712318+00:00,2026-05-07T22:28:23.358216+00:00,2026-05-07T22:28:23.832093+00:00,2026-05-07T22:28:24.340627+00:00


## 3.3 Persist results and export best-model headline probabilities

In [4]:
# Save Stage 3 metrics to results/nlp_results.csv.
results.save(RESULTS_DIR)

# Best run by F1 so Stage 4 can consume a single probability file.
nlp_results_df = results.nlp_df().copy()
best_row = nlp_results_df.sort_values("f1", ascending=False).iloc[0]

# Model ID and seed of the top run.
best_model_name = best_row["model"]
best_seed = int(best_row["seed"])
best_model_path = NLP_MODELS[best_model_name]

print("Best NLP run used for probability export:")
print(best_row.to_string())

# Re-train that best configuration and export per-headline probabilities.
nlp_probs_df = train_best_model_and_export_probs(
    model_name=best_model_name,
    model_path=best_model_path,
    seed=best_seed,
    text_df=text_df,
    cutoff_date=cutoff_date,
)
save_nlp_probs_df(nlp_probs_df)

print(f"Total NLP experiments: {len(results.nlp_results)}")
print(f"Saved per-headline probabilities to {ARTIFACTS_DIR / 'nlp_probs.parquet'}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Results saved to /home/hrilab/energy-analysis-pipeline/results/
Best NLP run used for probability export:
model                                          BERT-base
seed                                                   8
accuracy                                        0.395522
f1                                              0.354152
train_time_s                                     46.0128
infer_time_s                                    0.875992
n_test_samples                                      2546
wall_train_start_iso    2026-05-07T22:34:52.076695+00:00
wall_train_end_iso      2026-05-07T22:35:38.089519+00:00
wall_infer_start_iso    2026-05-07T22:35:38.935814+00:00
wall_infer_end_iso      2026-05-07T22:35:39.811812+00:00


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Total NLP experiments: 40
Saved per-headline probabilities to /home/hrilab/energy-analysis-pipeline/artifacts/nlp_probs.parquet
